In [26]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langgraph.graph import START,END,StateGraph
import operator
from typing import TypedDict,Annotated
from langchain.messages import SystemMessage,AIMessage,HumanMessage,AnyMessage

import requests
import streamlit as st

In [27]:
from tools.flight_tool import search_flights
from tools.tavily_tool import tavily_search

In [28]:
llm = ChatGroq(model = "llama-3.3-70b-versatile")

In [29]:
class TravelState(TypedDict):
    messages: Annotated[list[AnyMessage],operator.add]
    user_query : str
    flight_results : str
    hotel_results :str
    itinerary  : str
    llm_call : int


In [30]:
def flight_agent(state : TravelState):
    query = state["user_query"]
    flights_data = search_flights(query)
    return{
        "flight_results": flights_data,
        "messages": [AIMessage(content = f"Flight results fetched")],
        "llm_call":state.get("llm_call",0)+1
    }

In [31]:
def hotel_agent(state : TravelState):
    query =f"Best Hotels For {state["user_query"]}"
    hotel_data = tavily_search(query)
    return {
        "hotel_results" : hotel_data,
        "messages" : [AIMessage(content = f"hotel results Fetched")],
        "llm_call": state.get("llm_call",0)+1
    }

In [32]:
def itinerary_agent(state : TravelState):
    prompt = f"""
        Create a Tarvel Itinerary
        User Query : {state['user_query']}
        Flight Results : {state['flight_results']}
        Hotel Results : {state['hotel_results']}
        """
    response = llm.invoke([
        SystemMessage(content = "You are a Travel Expert Planner"),
        HumanMessage (content = prompt)
    ])

    return {
        "itinerary ": response.content,
        "message": [response],
        "llm_call": state.get('llm_call',0)+1
        }

In [33]:
def final_agent(state :TravelState):
    final_prompt = f""" Generate a Final Travel Response
    Flights : {state['flight_results']}
    Hotels : {state['hotel_results']}
    Itinerary  : {state['itinerary']}
    """

    response = llm.invoke([HumanMessage(content = final_prompt)])
    # Another way to call LLM
    # response = llm.invoke(final_prompt)


In [34]:
graph = StateGraph(TravelState)

graph.add_node("Flight_agent",flight_agent)
graph.add_node("Hotel_agent",hotel_agent)
graph.add_node ("itinerary_agent",itinerary_agent)
graph.add_node("Final_agent",final_agent)


graph.add_edge(START,"Flight_agent")
graph.add_edge("Flight_agent","Hotel_agent")
graph.add_edge("Hotel_agent","itinerary_agent")
graph.add_edge("itinerary_agent","Final_agent")
graph.add_edge ("Final_agent",END)

app = graph.compile()
# app.get_graph().print_ascii()

In [35]:
if __name__ == "__main__":
    config = {
        "configurable":{
            "thread_id" :"user_name is sukesh"
        }
    }

    user_input = input("Enter Your Travel Plan: ")

    result = app.invoke(
        {
            "messages": [HumanMessage(content = user_input)],
            "user_query" : user_input,
            "flight_results": "",
            "Hotel_results": "",
            "itinerary": "",
            "llm_call" : 0
        },config = config
        )
    # for msg in result['messages']:
    #     print(msg.content)
    print(result)


{'messages': [HumanMessage(content='Japan', additional_kwargs={}, response_metadata={}), AIMessage(content='Flight results fetched', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), AIMessage(content='hotel results Fetched', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'user_query': 'Japan', 'flight_results': '\n                           Airline : LOT,\n                           Departure: Beijing Capital International,\n                           Arrival : unknown,\n                           Status : scheduled\n                           \n\n                           Airline : Air China,\n                           Departure: Beijing Capital International,\n                           Arrival : unknown,\n                           Status : scheduled\n                           \n\n                           Airline : Air China,\n                           Departure: Beijing Capital International,\n            

## Add Streamlit to use in web Application

In [36]:
st.title("Travel Agent")
topic = st.text_input("Input the Tarvel Plan")

if topic:
    response = app.invoke(
        {
            "messages": [HumanMessage(content = user_input)],
            "user_query" : user_input,
            "flight_results": "",
            "Hotel_results": "",
            "Itinerary": "",
            "llm_call" : 0
        },config = config
        )
    st.write(response)
    # result = requests.post("http://127.0.0.1:8000")


2026-06-20 19:30:31.527 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 19:30:31.528 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 19:30:31.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 19:30:31.533 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 19:30:31.536 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 19:30:31.538 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 19:30:31.540 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-20 19:30:31.543 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar